In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
token = os.getenv("HF_TOKEN")

In [9]:
CORPUS_PATH = "./corpus"

In [13]:
import re
from collections import Counter

counter = Counter()

with open(f"{CORPUS_PATH}/train.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = re.findall(r"\w+|[^\w\s]", line)
        counter.update(tokens)

unique = len(counter)

print(f"Total tokens: {sum(counter.values()):,}")
print(f"Unique tokens: {unique:,}")

Total tokens: 5,086,331
Unique tokens: 120,287


In [14]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

In [28]:
VOCAB_SIZE = tokenizer.vocab_size

In [19]:
text = "Hello world!"
ids = tokenizer.encode(text)

In [ ]:
class WhiteSpaceTokenizer:
    def __init__(self, train_path: str, vocab_size: int):
        self.vocab_size = vocab_size
        self.vocab = {}
        self._create_vocab(train_path)

    def _create_vocab(self, train_path: str):
        counter = Counter()
        with open(train_path, "r", encoding="utf-8") as f:
            for line in f:
                tokens = line.strip().split()
                counter.update(tokens)

        # Keep only the most common tokens up to the specified vocab size
        counter = counter.most_common(self.vocab_size)

        self.vocab = {token: idx for idx, (token, _) in enumerate(counter)}
        self.vocab_size = len(self.vocab)

    def encode(self, text):
        return text.split()

    def decode(self, tokens):
        return " ".join(tokens)

In [ ]:
ws_tokenizer = WhiteSpaceTokenizer(f"{CORPUS_PATH}/train.txt", VOCAB_SIZE)

In [26]:
ws_tokenizer.vocab_size

50257

In [29]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input=f"{CORPUS_PATH}/train.txt",
    model_prefix="sp_check",
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=1.0,
    hard_vocab_limit=False
)

sp = spm.SentencePieceProcessor(
    model_file="sp_check.model"
)

actual_vocab = sp.get_piece_size()

print(f"Requested vocab: {VOCAB_SIZE}")
print(f"Actual vocab: {actual_vocab}")
print(f"Target achievable: {actual_vocab == VOCAB_SIZE}")

I0000 00:00:1792002525.931515   34300 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: ./corpus/train.txt
  input_format: 
  model_prefix: sp_check
  model_type: UNIGRAM
  vocab_size: 50257
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_

Requested vocab: 50257
Actual vocab: 50257
Target achievable: True
